# 03_finetune.ipynb — Fine-tuning Validation

**Project:** SentinelFatal2 — Foundation Model for Fetal Distress Prediction  
**Step:** 4 — Fine-tuning for Acidemia Classification  
**Source:** arXiv:2601.06149v1, Section II-E  
**SSOT:** `docs/work_plan.md`, Part ו שלב 4

---

## Validation Checks (V4.1 - V4.8)

This notebook validates all requirements for Agent 4 before handoff to Agent 5.

**Critical constraints:**
- ❌ **NO test.csv access** (data leakage prevention — G4.5, G4.9, V4.5)
- ✅ Class weights from train.csv only (~[1.0, 3.9])
- ✅ Differential LR: backbone 1e-5, head 1e-4
- ✅ AUC per-recording with max aggregation (P7 fix)
- ✅ Gradient clipping max_norm=1.0
- ✅ Early stopping patience=15 on val AUC

## Setup

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.model.patchtst import PatchTST, load_config
from src.model.heads import ClassificationHead
from src.data.dataset import FinetuneDataset, build_finetune_loaders
from src.train.utils import compute_recording_auc, sliding_windows
from src.train.finetune import compute_class_weights

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
device = torch.device('cpu')
print(f"Device: {device}")

## Load Configuration

In [ ]:
config_path = project_root / "config" / "train_config.yaml"
cfg = load_config(config_path)

print("Fine-tuning configuration:")
ftcfg = cfg['finetune']
for k, v in ftcfg.items():
    print(f"  {k}: {v}")

## V4.1 — Class Weights Calculation

**Requirement:** Class weights computed from train.csv only: ~[1.0, 3.9]  
**Method:** P8 fix (deviation S6.1) — use class_weight, not oversampling

In [ ]:
train_csv = project_root / "data" / "splits" / "train.csv"

# V4.1 Check
class_weights = compute_class_weights(train_csv)

# Validate
assert class_weights.shape == (2,), f"Expected shape (2,), got {class_weights.shape}"
assert class_weights[0] == 1.0, f"Weight for class 0 should be 1.0, got {class_weights[0]}"
assert 3.5 <= class_weights[1] <= 4.5, f"Weight for class 1 should be ~3.9, got {class_weights[1]}"

print(f"\n[V4.1] PASS - Class weights: {class_weights.tolist()}")
print(f"  Expected: [1.0, ~3.9]")
print(f"  Computed from train.csv only (NO val/test data)")

## V4.2 — Single Batch Training Test

**Requirement:** Training loop (forward → loss → backward → step) runs on 1 batch without errors

In [ ]:
# Build model with pretrained backbone (if available) or random init
model = PatchTST(cfg)
pretrain_ckpt = project_root / "checkpoints" / "pretrain" / "best_pretrain.pt"
if pretrain_ckpt.exists():
    model.load_state_dict(torch.load(pretrain_ckpt, map_location=device))
    print(f"Loaded pretrained checkpoint: {pretrain_ckpt}")
else:
    print("WARNING: No pretrained checkpoint found, using random init")

# Replace head
d_in = cfg['data']['n_patches'] * cfg['model']['d_model'] * cfg['data']['n_channels']
model.replace_head(ClassificationHead(d_in=d_in, n_classes=2, dropout=0.2))
model = model.to(device)

# Build data loader
train_loader, _ = build_finetune_loaders(
    train_csv=project_root / "data" / "splits" / "train.csv",
    val_csv=project_root / "data" / "splits" / "val.csv",
    processed_root=project_root / "data" / "processed",
    batch_size=4,
)

# Setup optimizer and loss
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
backbone_params = list(model.patch_embed.parameters()) + list(model.encoder.parameters())
head_params = list(model.head.parameters())
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params, 'lr': 1e-4},
], weight_decay=1e-2)

# V4.2 Check: Single batch forward-backward-step
model.train()
batch_x, batch_y = next(iter(train_loader))
batch_x, batch_y = batch_x.to(device), batch_y.to(device)

optimizer.zero_grad()
logits = model(batch_x)
loss = criterion(logits, batch_y)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

# Validate
assert not torch.isnan(loss), "Loss is NaN!"
assert not torch.isinf(loss), "Loss is Inf!"
assert loss.item() > 0, "Loss should be positive"

print(f"\n[V4.2] PASS - Single batch training successful")
print(f"  Batch shape: {batch_x.shape}, Labels: {batch_y.shape}")
print(f"  Logits shape: {logits.shape}")
print(f"  Loss: {loss.item():.6f}")

## V4.3 — AUC Function Sanity Check

**Requirement:** `compute_recording_auc()` returns value in [0, 1] on val set

In [ ]:
# V4.3 Check: Compute AUC on validation set (small stride for speed)
val_csv = project_root / "data" / "splits" / "val.csv"
processed_root = project_root / "data" / "processed"

model.eval()
val_auc = compute_recording_auc(
    model, val_csv, processed_root, stride=60, device='cpu'
)

# Validate
assert 0.0 <= val_auc <= 1.0, f"AUC should be in [0, 1], got {val_auc}"

print(f"\n[V4.3] PASS - AUC computation successful")
print(f"  Val AUC: {val_auc:.6f}")
print(f"  (Using stride=60 for speed; official eval uses stride=1)")

## V4.4 — AUC Per-Recording Verification (Code Review)

**Requirement:** AUC is per-recording with max aggregation, NOT per-window  
**P7 Fix:** Training unit = window, Evaluation unit = recording

In [ ]:
import inspect

# V4.4 Check: Verify max aggregation in compute_recording_auc source
source = inspect.getsource(compute_recording_auc)

# Check for key patterns
assert 'max(scores)' in source, "Missing max aggregation!"
assert 'recording_score' in source, "Missing recording-level score!"
assert 'roc_auc_score' in source, "Missing ROC-AUC computation!"

# Print relevant code snippet
print("[V4.4] PASS - AUC computed per-recording with max aggregation")
print("\nKey code snippet:")
for line in source.split('\n'):
    if 'max(scores)' in line or 'recording_score' in line:
        print(f"  {line.strip()}")

## V4.5 — Test Data Verification

**Requirement:** Test set (test.csv) doesn't appear anywhere in code  
**Critical:** This is a data leakage check (G4.5, G4.9)

In [ ]:
# V4.5 Check: Grep for test.csv in production code (NOT validation notebooks)
# Note: We check only src/ files, not notebooks (which contain documentation about test.csv)
files_to_check = [
    'src/data/dataset.py',
    'src/train/finetune.py',
    'src/train/utils.py',
]

violations = []
for fname in files_to_check:
    fpath = project_root / fname
    if fpath.exists():
        content = fpath.read_text(encoding='utf-8')
        if 'test.csv' in content:
            violations.append(fname)

# Validate
assert len(violations) == 0, f"CRITICAL: test.csv found in: {violations}"

print("[V4.5] PASS - NO test.csv access detected in production code")
print(f"  Checked files: {len(files_to_check)}")
print(f"  Files: {', '.join(files_to_check)}")
print("  Data leakage prevention: OK")

## V4.6 — Differential Learning Rate Verification

**Requirement:** Differential LR: backbone=1e-5, head=1e-4

In [ ]:
# V4.6 Check: Inspect optimizer param groups
print("[V4.6] Optimizer parameter groups:")
for i, group in enumerate(optimizer.param_groups):
    n_params = sum(p.numel() for p in group['params'])
    lr = group['lr']
    name = 'backbone' if i == 0 else 'head'
    print(f"  Group {i} ({name}): lr={lr:.2e}, params={n_params:,}")

# Validate
lr_backbone = optimizer.param_groups[0]['lr']
lr_head = optimizer.param_groups[1]['lr']

assert lr_backbone == 1e-5, f"Backbone LR should be 1e-5, got {lr_backbone}"
assert lr_head == 1e-4, f"Head LR should be 1e-4, got {lr_head}"
assert lr_head > lr_backbone, "Head LR should be higher than backbone LR"

print(f"\n[V4.6] PASS - Differential LR verified")
print(f"  Backbone LR: {lr_backbone:.2e} (prevents catastrophic forgetting)")
print(f"  Head LR:     {lr_head:.2e} (faster adaptation)")

## V4.7 — Gradient Clipping Verification

**Requirement:** Gradient clipping max_norm=1.0 active

In [ ]:
import inspect

# V4.7 Check: Verify gradient clipping in finetune.py source
from src.train import finetune
source = inspect.getsource(finetune.run_epoch)

# Check for gradient clipping call
assert 'clip_grad_norm_' in source, "Missing gradient clipping!"
assert 'max_norm' in source, "Missing max_norm parameter!"

# Check config value
clip_norm = cfg['finetune']['gradient_clip']
assert clip_norm == 1.0, f"gradient_clip should be 1.0, got {clip_norm}"

print("[V4.7] PASS - Gradient clipping verified")
print(f"  max_norm: {clip_norm}")
print("  Applied in: run_epoch() before optimizer.step()")

## V4.8 — Early Stopping Logic Check

**Requirement:** Early stopping on val AUC with patience=15

In [ ]:
import inspect

# V4.8 Check: Verify early stopping in finetune.py
source = inspect.getsource(finetune.train)

# Check for early stopping patterns
assert 'best_val_auc' in source, "Missing best_val_auc tracking!"
assert 'patience_ctr' in source or 'patience_counter' in source, "Missing patience counter!"
assert 'val_auc > best_val_auc' in source, "Missing AUC improvement check!"

# Check config value
patience = cfg['finetune']['patience']
assert patience == 15, f"patience should be 15, got {patience}"

print("[V4.8] PASS - Early stopping logic verified")
print(f"  Patience: {patience} epochs")
print("  Metric: validation AUC (per-recording)")
print("  Direction: maximize")

## Summary — All Validation Checks

If all cells above passed, Agent 4 implementation is complete and ready for handoff.

In [ ]:
print("="*60)
print("AGENT 4 VALIDATION SUMMARY")
print("="*60)
print("[V4.1] Class weights computed from train.csv: PASS")
print("[V4.2] Single batch training test: PASS")
print("[V4.3] AUC function returns [0,1]: PASS")
print("[V4.4] AUC per-recording (max aggregation): PASS")
print("[V4.5] NO test.csv access (data leakage check): PASS")
print("[V4.6] Differential LR (backbone=1e-5, head=1e-4): PASS")
print("[V4.7] Gradient clipping (max_norm=1.0): PASS")
print("[V4.8] Early stopping (patience=15, metric=val_auc): PASS")
print("="*60)
print("\nAll validation checks passed!")
print("Ready for handoff to Agent 5.")
print("\nAgent 6 will execute the actual training.")

## Handoff Checklist

Agent 5 (Inference & Alerting) will need:

- ✅ `src/model/patchtst.py` — Model architecture
- ✅ `src/model/heads.py` — ClassificationHead
- ✅ `src/train/finetune.py` — Training script (for checkpoint loading pattern)
- ✅ `src/train/utils.py` — `compute_recording_auc()` (inference + AUC logic)
- ⏳ `checkpoints/finetune/best_finetune.pt` — Trained weights (created by Agent 6)

**Pre-Run Gates (G4.1-G4.9):** All verified ✅  
**Definition of Done (V4.1-V4.8):** All validated ✅  
**Test data access:** ZERO (critical constraint satisfied) ✅